# Logistics Pulse - Data Exploration

First look at the 9 raw Olist tables before any cleaning. For each table:
shape, structure, missing values, duplicates - then table-specific checks
where the business question requires it.

## Setup

Load all 9 raw Olist tables into a dict, keyed by table name, for easy
iteration below.

In [1]:
from pathlib import Path
import pandas as pd

RAW_DATA_DIR = Path("../data/raw")

raw = {
    "customers": pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv"),
    "geolocation": pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv"),
    "orders": pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv"),
    "order_items": pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv"),
    "order_payments": pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv"),
    "order_reviews": pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv"),
    "sellers": pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv"),
    "category_translation": pd.read_csv(RAW_DATA_DIR / "product_category_name_translation.csv"),
}

## Helper function

All 9 tables get the same five checks (`shape`, `info()`, missing values,
duplicates, `head()`) — defined once here instead of repeating the same
code nine times.

In [2]:
def explore(df: pd.DataFrame, name: str) -> None:
    """Print the standard first-look checks for one raw table."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print()
    df.info()
    print()
    print("missing values per column:")
    print(df.isnull().sum())
    print()
    print(f"duplicated rows: {df.duplicated().sum()}")
    print()
    print(df.head())

## customers

Reference table for the customer dimension.

In [3]:
explore(raw["customers"], "customers")

=== customers ===
shape: (99441, 5)

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB

missing values per column:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

duplicated rows: 0

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586e

**Findings:** 99,441 rows, 0 missing values, 0 duplicates. Clean reference
table, matches expectations - no action needed.

## geolocation

Maps zip code prefixes to coordinates. Needed later to calculate the
distance between sellers and customers. This table is expected to have
many duplicate rows per zip code - checking the scale below.

In [4]:
explore(raw["geolocation"], "geolocation")

=== geolocation ===
shape: (1000163, 5)

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB

missing values per column:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

duplicated rows: 261831

   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1          

**Findings:** 1 000 163 rows, 0 missing values, 261 831 exact duplicate
rows.

**Next**: check how many unique zip codes exist compared to the total
rows, to see how much repetition there is per zip code.

In [5]:
n_total = len(raw["geolocation"])
n_unique_zips = raw["geolocation"]["geolocation_zip_code_prefix"].nunique()

print(f"total rows: {n_total}")
print(f"unique zip codes: {n_unique_zips}")
print(f"average rows per zip code: {n_total / n_unique_zips:.1f}")

total rows: 1000163
unique zip codes: 19015
average rows per zip code: 52.6


**Findings:** 19 015 unique zip codes, about 52.6 rows per zip code on
average. This confirms the table needs to be aggregated (one row per zip
code, using average coordinates) before it can be used to calculate
distances.

## orders

Core table for the whole analysis - every KPI depends on it. Beyond the
standard checks, we look at two more things: the order status
distribution, and whether any timestamps look out of order (for example,
approval logged before purchase).

In [6]:
explore(raw["orders"], "orders")

=== orders ===
shape: (99441, 8)

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB

missing values per column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     17

**Findings:** 99 441 rows, 8 columns, 0 duplicates. Missing values grow
along the delivery timeline - consistent with orders dropping out at different
stages.

**Next**: checking the order status distribution. `delay_days` will
only be calculated for orders with status "delivered" - this shows how
many orders that covers, and how many are excluded.

In [7]:
raw["orders"]["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**Findings:** 8 order statuses confirmed. 96 478 orders (97%) are
"delivered" and get a `delay_days` value; the remaining 2 963 (3%) are
excluded from delay analysis. As an aside - out of curiosity, not part of
the main analysis - checking below whether the missing dates line up with
these statuses before moving on to the date anomaly check.

In [8]:
raw["orders"].groupby("order_status")[["order_delivered_carrier_date", "order_approved_at", "order_delivered_customer_date"]].agg(lambda x: x.isnull().sum())

,order_delivered_carrier_date,order_approved_at,order_delivered_customer_date
order_status,,,
approved,2,0,2
canceled,550,141,619
created,5,5,5
delivered,2,14,8
invoiced,314,0,314
processing,301,0,301
shipped,0,0,1107
unavailable,609,0,609


**Findings:** Missing dates mostly follow order status - all 1 107
"shipped" orders miss a customer delivery date, as expected. Two
exceptions stand out: only 550/625 "canceled" orders miss a carrier date
(some were canceled after shipping), and 14 "delivered" orders are
missing an approval date despite the status implying full completion.

**Next**: Checking whether any order was approved before it was purchased -
logically impossible, and a red flag for data quality.

In [9]:
def anomaly_check(df: pd.DataFrame, col_earlier: str, col_later: str):
    """Return rows where col_later happened before col_earlier."""
    col_earlier = pd.to_datetime(df[col_earlier])
    col_later = pd.to_datetime(df[col_later])
    return df[col_earlier > col_later]

print(anomaly_check(raw["orders"], "order_purchase_timestamp", "order_approved_at").shape)
print(anomaly_check(raw["orders"], "order_approved_at", "order_delivered_carrier_date").shape)
print(anomaly_check(raw["orders"], "order_delivered_carrier_date", "order_delivered_customer_date").shape)

(0, 8)
(1359, 8)
(23, 8)


**Findings:** 0 orders approved before purchase, as expected. 1 359
orders (1.4%) reached the carrier before approval - unusually fast, but
minor relative to the dataset. 23 orders (0.02%) show delivery before
carrier handoff - a stronger red flag, since there's no reasonable
explanation for that. Decision on handling deferred to
`clean_orders.py`.

## order_items

One row per item in an order. Checking price distribution below (zeros,
extreme outliers) alongside the standard stats.

In [10]:
explore(raw["order_items"], "order_items")

=== order_items ===
shape: (112650, 7)

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB

missing values per column:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

duplicated rows: 0

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   

**Findings:** 112 650 rows, 0 missing values, 0 duplicates.

**Next**: checking the price distribution for zeros and outliers.

In [11]:
raw["order_items"]["price"].describe()

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

**Findings:** No zero-price items (min: 0.85). Prices are right-skewed -
75% of items cost 134.90 or less, but the max is 6 735, and std (183.63)
exceeds the mean (120.65). A small number of high-value items pull the
average up; most items are cheap.

## order_payments

Payment details per order - a single order can have multiple payment
rows (e.g. split across installments). Checking the standard stats
first, then how many distinct payment types exist.

In [12]:
explore(raw["order_payments"], "order_payments")

=== order_payments ===
shape: (103886, 5)

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB

missing values per column:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

duplicated rows: 0

                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708

**Findings:** 103 886 rows, 0 missing values, 0 duplicates.

**Next**:
checking how many distinct payment types exist and how often each
occurs.

In [13]:
raw["order_payments"]["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

**Findings:** 5 payment types confirmed. Credit card dominates (76 795,
74%), followed by boleto (19 784, 19%). 3 rows have an undefined payment
type - a tiny edge case, not worth digging into further.

## order_reviews

Customer reviews per order. Checking two things beyond the standard
stats: how many review comments are missing, and the distribution of
review scores (typically skewed toward high ratings - if not, that's a
signal worth noting).

In [14]:
explore(raw["order_reviews"], "order_reviews")

=== order_reviews ===
shape: (99224, 7)

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB

missing values per column:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

duplicated rows: 0

                          review_id                    

**Findings:** 99 224 rows, 0 duplicates. Most reviews have no written
feedback - 87 656 (88%) are missing a title, and 58 247 (59%) are missing
a message. This lines up with a common pattern: people mostly write
comments at the extremes (very happy or very unhappy), not for average
experiences.

**Next**: checking the review_score distribution.

In [15]:
raw["order_reviews"]["review_score"].value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

**Findings:** Scores skew heavily positive - 77% are 4 or 5 stars. But
the shape isn't a smooth decline: 1-star reviews (11.5%) outnumber 2-star
and 3-star combined (11.4%). When customers are unhappy, they rarely
rate "moderately low" - they go straight to 1 star.

## products

Product catalog. Known from planning to have inconsistent column
spellings (e.g. "lenght" instead of "length") and some missing category
values - checking both below alongside the standard stats.

In [17]:
explore(raw["products"], "products")

=== products ===
shape: (32951, 9)

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB

missing values per column:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty          

**Findings:** 32 951 rows, 0 duplicates. Missing values cluster in two
groups: 610 rows missing category/name/description/photos together
(incomplete listings), 2 rows missing physical dimensions. Confirms the
known "lenght" column typo - left as-is, fixed later in `clean_products.py`.

## sellers

Seller reference table. Standard checks only - no specific concerns
flagged in planning for this table.

In [18]:
explore(raw["sellers"], "sellers")

=== sellers ===
shape: (3095, 4)

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB

missing values per column:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

duplicated rows: 0

                          seller_id  seller_zip_code_prefix  \
0  3442f8959a84dea7ee197c632cb2df15                   13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4  51a04a8a6bdcb23deccc82b0b8

**Findings:** 3 095 rows, 0 missing values, 0 duplicates. Clean reference
table, no issues.

## category_translation

Small lookup table translating category names from Portuguese to
English. Standard checks only.

In [19]:
explore(raw["category_translation"], "category_translation")

=== category_translation ===
shape: (71, 2)

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB

missing values per column:
product_category_name            0
product_category_name_english    0
dtype: int64

duplicated rows: 0

    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor


**Findings:** 71 rows, 0 missing values, 0 duplicates. Clean lookup
table, no issues.

## Cross-table checks

With all 9 tables explored individually, this section checks how they
connect: order_id consistency between orders and order_items,
referential integrity for the relationships this analysis actually uses,
and zip code coverage in geolocation.

In [27]:
orders_ids = set(raw["orders"]["order_id"])
order_items_ids = set(raw["order_items"]["order_id"])
orphaned_order_ids = orders_ids - order_items_ids
len(orphaned_order_ids)

775

**Findings:** 775 orders have no matching order_items rows.

**Next:** checking whether this lines up with order status.

In [28]:
raw["orders"][raw["orders"]["order_id"].isin(orphaned_order_ids)]["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

**Findings:** Not a data quality issue - it lines up with order status.
603/609 "unavailable" orders (99%) have no items, vs only 164/625 (26%)
of canceled ones.

**Next:** checking referential integrity only for
relationships this analysis uses - order_items to products/sellers, and
order_reviews to orders.

In [35]:
def check_referential_integrity(child_df, child_col, parent_df, parent_col):
    """Return values from child_col that have no match in parent_col."""
    child_ids = set(child_df[child_col])
    parent_ids = set(parent_df[parent_col])
    return child_ids - parent_ids

print(len(check_referential_integrity(raw["order_items"], "product_id", raw["products"], "product_id")))
print(len(check_referential_integrity(raw["order_items"], "seller_id", raw["sellers"], "seller_id")))
print(len(check_referential_integrity(raw["order_reviews"], "order_id", raw["orders"], "order_id")))


0
0
0


**Findings:** 0 orphaned rows across all three relationships checked
(order_items to products, order_items to sellers, order_reviews to
orders). Safe to join on these keys later without silently losing rows.

**Next:** checking geolocation coverage - how many customer and seller zip
codes have no match in geolocation.

In [36]:
customer_zips = set(raw["customers"]["customer_zip_code_prefix"])
seller_zips = set(raw["sellers"]["seller_zip_code_prefix"])
geolocation_zips = set(raw["geolocation"]["geolocation_zip_code_prefix"])

unmatched_customer_zips = customer_zips - geolocation_zips
unmatched_seller_zips = seller_zips - geolocation_zips

print(f"customer zip codes with no match: {len(unmatched_customer_zips)} / {len(customer_zips)} ({len(unmatched_customer_zips)/len(customer_zips):.1%})")
print(f"seller zip codes with no match: {len(unmatched_seller_zips)} / {len(seller_zips)} ({len(unmatched_seller_zips)/len(seller_zips):.1%})")

customer zip codes with no match: 157 / 14994 (1.0%)
seller zip codes with no match: 7 / 2246 (0.3%)


**Findings:** 157 of 14 994 customer zip codes (1.0%) and 7 of 2 246
seller zip codes (0.3%) have no match in geolocation - small gaps, will
result in a few missing distance calculations later, not a major issue.